# Trabalho residual de inferência ao final da captura

Campanha: `battery_mas-single_20260708_104924`. Este notebook usa exclusivamente os `metrics.json` das cinco runs de 1, 2, 3, 4, 5, 10, 15, 20 e 30 FPS.

A unidade experimental para IC95% e testes globais é a execução completa (n=5 por FPS). As 184 passagens são observações internas de cada run, não réplicas independentes do tratamento.

## Definições

No instante `last_capture = last_image_capture_time`:

- `active_inferences = count(prediction_start <= last_capture < prediction_final)`;
- `not_started_inferences = count(prediction_start > last_capture)`;
- `residual_inferences = active_inferences + not_started_inferences = count(prediction_final > last_capture)`;
- `residual_fraction = residual_inferences / suitable_images`, ausente quando `suitable_images = 0`;
- `prediction_drain_s = max(0, max(prediction_final por imagem) - last_capture)`;
- `post_capture_latency_s = weight_prediction_final da passagem - last_capture`;
- `finalization_overhead_s = weight_prediction_final da passagem - max(last_capture, max(prediction_final por imagem))`.

`residual_inferences` é denominado **residual inference workload at the end of capture**. `not_started_inferences` representa **inferences not yet started at the prediction stage** no instante da última captura.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

campaign_dir = Path.cwd().resolve()
if not (campaign_dir / '_manifest.txt').exists():
    candidate = Path('power_runs/battery_mas-single_20260708_104924').resolve()
    if candidate.exists():
        campaign_dir = candidate
    else:
        raise FileNotFoundError('Diretório da campanha não encontrado.')

sys.path.insert(0, str(campaign_dir))
from residual_inference_workload import run_analysis

output_dir = campaign_dir / 'residual_analysis'
results = run_analysis(campaign_dir, output_dir)
print(f'Campanha: {campaign_dir}')
print(f'Saídas: {output_dir}')

## A. Nível da passagem

A tabela completa é exportada para `residual_analysis/residual_by_passage.csv`.

In [ ]:
display(results['passage'].head(12))
print('Linhas:', len(results['passage']))
print('Passagens com suitable_images = 0 e residual_fraction ausente:',
      int(results['passage']['suitable_images'].eq(0).sum()))

## B. Nível da run

Cada linha abaixo resume uma execução completa. O P95 é o quantil empírico com interpolação linear. `not_started_share_of_residual` é a razão entre as somas de `not_started_inferences` e `residual_inferences` na run, com `NaN` quando a soma residual é zero.

In [ ]:
display(results['run'].round(6))

## C. Nível de FPS

A primeira tabela apresenta a decomposição compacta solicitada; cada valor é a média das cinco métricas de run no FPS correspondente. Em seguida, o formato longo preserva, para todas as métricas, a média, o desvio padrão amostral e o IC95% t de Student das cinco runs.

In [ ]:
display(results['decomposition_summary'].round(6))
display(results['fps'].round(6))

## Verificações de integridade e testes globais

A execução é interrompida pelo módulo se qualquer verificação obrigatória falhar. Isso inclui `active_inferences <= 1`, a decomposição exata `active_inferences + not_started_inferences == residual_inferences`, `prediction_start <= prediction_final`, `finalization_overhead_s >= 0` e a identidade `post_capture_latency_s ≈ prediction_drain_s + finalization_overhead_s` com tolerância absoluta de 1e-6 s. O Kruskal–Wallis usa apenas as 45 métricas por run, é omitido para valores determinísticos ou sem variação suficiente e não dispara pós-hoc.

In [ ]:
display(results['integrity'])
display(results['kruskal_wallis'].round(6))
print('Problemas de timestamp/esquema encontrados:',
      len(results['timestamp_and_schema_issues']))

## Figuras

As figuras existentes são mantidas; nenhuma nova figura foi criada para a decomposição. As barras de erro são IC95% no nível da run. A segunda figura seleciona `symlog` quando a amplitude dos resultados comprimir as configurações menores.

In [ ]:
display(Image(filename=str(results['paths']['residual_png'])))
display(Image(filename=str(results['paths']['drain_png'])))
print('Escala da figura de drenagem:', results['figure_info']['prediction_drain_scale'])

## Relatório e artefatos

In [ ]:
display(Markdown(results['paths']['report'].read_text(encoding='utf-8')))
for name, path in results['paths'].items():
    print(f'{name}: {path}')